# 실습 7: 학습용과 시험용으로 나누기
- 상황: 아직 검사하지 않은 흐름의 결과를 맞혀보려 한다
- 목표: 답을 아는 기록과 모르는 척할 기록을 나눈다

## Step 0. 정제본 불러오기

In [1]:
import pandas as pd

# 이 노트북은 day03/lab07_train-test-split 에, 파일은 day02 실습 결과물 폴더에 있다
# 두 칸 위로 올라갔다가(../..) day02 쪽으로 들어간다
df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

print("행/열:", df.shape)
print()
print(df["result"].value_counts())

행/열: (1567, 51)

result
양품    1463
불량     104
Name: count, dtype: int64


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 모델을 만들 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 지도학습 | 답이 붙어 있는 기록으로 규칙을 찾게 하는 방식. 우리 데이터의 검사 결과가 그 답이다 |
| 분류 | 둘 중 어느 쪽인지 맞히는 문제. 양품이냐 불량이냐 |
| 학습용 | 답을 보여주고 규칙을 찾게 할 몫 |
| 시험용 | 답을 숨겨두고 실력을 재는 데 쓸 몫 |
| 과적합 | 학습용을 통째로 외워버려 처음 보는 기록은 못 맞히는 상태 |
| 일반화 | 그 반대. 처음 보는 기록에도 통하는 상태. 우리가 원하는 것 |
| 클래스 불균형 | 한쪽이 드문 상태. 여기서는 불량이 약 6.6%뿐이다 |
| 층화추출 | 나눌 때 드문 쪽 비율을 양쪽에 똑같이 맞춰주는 방식 |

## Step 2. 남은 빈칸 채우기

In [2]:
# 센서 열에 남은 빈칸을 세고, 그 열의 중앙값으로 채운다

센서열 = [c for c in df.columns if c.startswith("sensor_")]

빈칸수 = df[센서열].isna().sum()          # 열마다 빈칸 개수
전체칸 = df[센서열].size                   # size - 행 x 열, 즉 칸의 총 개수

print("[채우기 전]")
print("센서 열:", len(센서열), "개")
print("빈칸이 있는 열:", int((빈칸수 > 0).sum()), "개")
print("빈칸 총 개수:", int(빈칸수.sum()), "/ 전체 칸:", 전체칸)
print("전체 칸 중 비율:", round(빈칸수.sum() / 전체칸 * 100, 3), "%")

print("\n빈칸이 많은 열 5개:")
print(빈칸수.sort_values(ascending=False).head(5))

# median() - 중앙값. 값을 크기순으로 세웠을 때 한가운데 값
# 평균과 달리 유난히 크거나 작은 값 몇 개에 잘 흔들리지 않는다
중앙값 = df[센서열].median()

# fillna(중앙값) - 빈칸을 그 열의 중앙값으로 채운다
# result 열은 센서열에 없으므로 건드리지 않는다
df[센서열] = df[센서열].fillna(중앙값)

print("\n[채운 뒤]")
print("센서 열 빈칸:", int(df[센서열].isna().sum().sum()), "개")
print("표 전체 빈칸:", int(df.isna().sum().sum()), "개")
print("result 열 그대로:", df["result"].value_counts().to_dict())
print("행/열:", df.shape)

[채우기 전]
센서 열: 50 개
빈칸이 있는 열: 48 개
빈칸 총 개수: 1539 / 전체 칸: 78350
전체 칸 중 비율: 1.964 %

빈칸이 많은 열 5개:
sensor_248    715
sensor_552    260
sensor_551    260
sensor_091     51
sensor_080     24
dtype: int64

[채운 뒤]
센서 열 빈칸: 0 개
표 전체 빈칸: 0 개
result 열 그대로: {'양품': 1463, '불량': 104}
행/열: (1567, 51)


## Step 3. 정답표를 숫자로 바꾸기

In [3]:
# result 열은 "양품"·"불량"이라는 글자다. 모델은 글자로 학습하지 못한다.
# == 로 비교하면 참/거짓이 되고, astype(int)가 참을 1, 거짓을 0으로 바꾼다
df["불량여부"] = (df["result"] == "불량").astype(int)

# 1의 개수가 앞에서 본 불량 건수와 같아야 한다
print(df["불량여부"].value_counts())

불량여부
0    1463
1     104
Name: count, dtype: int64


## Step 4. 입력과 정답으로 가르기

In [4]:
# 입력 - 센서 열만. result 와 불량여부는 절대 들어가면 안 된다
센서열 = [c for c in df.columns if c.startswith("sensor_")]
X = df[센서열]

# 정답 - 맞혀야 할 것
y = df["불량여부"]

print("입력:", X.shape)
print("정답:", y.shape)

입력: (1567, 50)
정답: (1567,)


## Step 5. 학습용과 시험용으로 나누기

In [5]:
# train_test_split - 표를 학습용과 시험용 두 몫으로 갈라준다
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 시험용으로 떼어둘 비율 (20%)
    random_state=42,      # 무작위로 섞되, 다시 실행해도 같게 나오도록 고정
    stratify=y            # 불량 비율을 양쪽에 똑같이 맞춰서 나눈다
)

print("학습용:", X_train.shape)
print("시험용:", X_test.shape)

학습용: (1253, 50)
시험용: (314, 50)


### 문법 노트 - 나누기

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| train_test_split(X, y) | 입력과 정답을 같은 기준으로 두 몫씩 갈라준다 | 답을 숨겨둔 몫이 있어야 실력을 잰다 |
| test_size=0.2 | 시험용으로 뗄 비율 | 20%면 300건 정도 남는다 |
| random_state=42 | 섞는 방식을 고정 | 안 넣으면 돌릴 때마다 결과가 달라져 비교가 안 된다 |
| stratify=y | 정답 비율을 양쪽에 맞춰 나눈다 | 불량이 6.6%뿐이라 안 맞추면 한쪽에 몰린다 |

**돌려주는 것이 네 덩어리인 순서에 주의.**<br>
X_train, X_test, y_train, y_test 순서다. 입력 둘이 먼저, 정답 둘이 나중.<br>
순서를 바꿔 받으면 오류 없이 실행되면서 결과만 이상해진다.

In [6]:
# 네 덩어리가 제대로 나뉘었는지 한 표로 확인한다

# X_train과 y_train은 같은 행을 가리킨다. 불량 건수는 짝이 되는 정답(y)에서 센다
덩어리 = [
    ("X_train", len(X_train), y_train),
    ("X_test",  len(X_test),  y_test),
    ("y_train", len(y_train), y_train),
    ("y_test",  len(y_test),  y_test),
]

행 = []
for 이름, 행수, 정답 in 덩어리:
    불량 = int((정답 == 1).sum())
    행.append({
        "덩어리": 이름,
        "행 수": 행수,
        "불량 건수": 불량,
        "불량 비율(%)": round(불량 / 행수 * 100, 2),
    })

# 비교 기준으로 원본 전체도 한 줄 넣는다
행.append({
    "덩어리": "원본 전체 (y)",
    "행 수": len(y),
    "불량 건수": int((y == 1).sum()),
    "불량 비율(%)": round((y == 1).mean() * 100, 2),
})

확인표 = pd.DataFrame(행).set_index("덩어리")

print("학습용 + 시험용 =", len(X_train) + len(X_test), "/ 원본", len(df))
print("불량 합계:", int((y_train == 1).sum()) + int((y_test == 1).sum()), "/ 원본", int((y == 1).sum()))
확인표

학습용 + 시험용 = 1567 / 원본 1567
불량 합계: 104 / 원본 104


,행 수,불량 건수,불량 비율(%)
덩어리,,,
X_train,1253,83,6.62
X_test,314,21,6.69
y_train,1253,83,6.62
y_test,314,21,6.69
원본 전체 (y),1567,104,6.64


[나눈 결과]<br>
학습용 : [1253]건 (불량 [83]건, [6.62]%)<br>
시험용 : [314]건 (불량 [21]건, [6.69]%)<br>
전체   : [1567]건 (불량 [104]건, [6.64]%)<br>
시험용 불량이 [21]건뿐이다.

## Step 7. 첫 예측 한 번 돌려보기

In [7]:
# 가장 기본적인 분류 모델 하나로 학습하고 시험용을 예측해본다

# 결정트리 - "이 값이 얼마보다 크면 이쪽" 식의 질문을 이어 붙여 가르는 모델
# 값의 크기를 그대로 써도 되어서, 표준화 없이 바로 쓸 수 있다
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)   # 같은 결과가 나오도록 고정

# fit - 학습용 입력과 정답을 보여주고 규칙을 찾게 한다
model.fit(X_train, y_train)

# predict - 시험용 입력만 주고 답을 맞혀보게 한다 (정답은 안 보여준다)
y_pred = model.predict(X_test)

print("1) 예측 결과 개수:", len(y_pred), "개")
print("2) 불량이라고 예측한 건수:", int((y_pred == 1).sum()), "건")
print("3) 시험용의 실제 불량:", int((y_test == 1).sum()), "건")

print("\n[예측한 값의 종류별 개수]")
print(pd.Series(y_pred).value_counts().rename({0: "양품(0)", 1: "불량(1)"}))

1) 예측 결과 개수: 314 개
2) 불량이라고 예측한 건수: 31 건
3) 시험용의 실제 불량: 21 건

[예측한 값의 종류별 개수]
양품(0)    283
불량(1)     31
Name: count, dtype: int64


---
## 직접 해보기 (도전) - 비율을 안 맞추고 나누면

- 상황: stratify 를 넣어야 한다고 배웠지만, 안 넣으면 실제로 얼마나 달라지는지는 모른다
- 할 일: stratify 없이 여러 번 나눠보고 시험용 불량 건수가 얼마나 흔들리는지 본다
- 결과물: 다섯 줄짜리 비교표 1개

In [8]:
# stratify를 넣은 경우와 뺀 경우를 나란히 비교한다
# 앞에서 만든 X_train, X_test, y_train, y_test는 건드리지 않는다 (다른 이름으로 받는다)

전체비율 = (y == 1).mean() * 100

행 = []
for 씨앗 in [0, 1, 2, 3, 4]:
    # stratify 없이 — 그냥 무작위로 자른다
    _, _, _, y시험_없음 = train_test_split(
        X, y, test_size=0.2, random_state=씨앗
    )
    # stratify=y 넣고 — 불량 비율을 맞춰서 자른다
    _, _, _, y시험_있음 = train_test_split(
        X, y, test_size=0.2, random_state=씨앗, stratify=y
    )

    행.append({
        "random_state": 씨앗,
        "없음 불량건수": int((y시험_없음 == 1).sum()),
        "없음 불량비율(%)": round((y시험_없음 == 1).mean() * 100, 2),
        "stratify 불량건수": int((y시험_있음 == 1).sum()),
        "stratify 불량비율(%)": round((y시험_있음 == 1).mean() * 100, 2),
    })

비교 = pd.DataFrame(행).set_index("random_state")

print("전체 불량 비율:", round(전체비율, 2), "%  (양쪽이 이 값에 얼마나 가까운지 본다)")
print("시험용 크기는 다섯 경우 모두 314건")
print()
print("[흔들린 폭]")
print(" stratify 없음 :", 비교["없음 불량건수"].min(), "~", 비교["없음 불량건수"].max(), "건")
print(" stratify 넣음 :", 비교["stratify 불량건수"].min(), "~", 비교["stratify 불량건수"].max(), "건")

print("\n앞에서 만든 것 그대로 — y_test 불량:", int((y_test == 1).sum()), "건")
비교

전체 불량 비율: 6.64 %  (양쪽이 이 값에 얼마나 가까운지 본다)
시험용 크기는 다섯 경우 모두 314건

[흔들린 폭]
 stratify 없음 : 13 ~ 26 건
 stratify 넣음 : 21 ~ 21 건

앞에서 만든 것 그대로 — y_test 불량: 21 건


,없음 불량건수,없음 불량비율(%),stratify 불량건수,stratify 불량비율(%)
random_state,,,,
0,13,4.14,21,6.69
1,20,6.37,21,6.69
2,20,6.37,21,6.69
3,19,6.05,21,6.69
4,26,8.28,21,6.69


### stratify 있음 vs 없음 (시험용 불량 건수와 비율)

| random_state | 있음 | 없음 |
|---|---|---|
| 0 | [21] (6.69%) | [13] (4.14%) |
| 1 | [21] (6.69%) | [20] (6.37%) |
| 2 | [21] (6.69%) | [20] (6.37%) |
| 3 | [21] (6.69%) | [19] (6.05%) |
| 4 | [21] (6.69%) | [26] (8.28%) |

흔들린 폭 : 있음 [21]~[21]건 / 없음 [13]~[26]건